## Week 2: Lasso, Ridge, and Elastic Net Regression

This notebook applies Week 2 concepts from DX799 Capstone Part 1 to the BRFSS 2015 diabetes dataset. Building on Week 1 findings where polynomial terms caused multicollinearity across all candidates, this week applies regularization techniques to address that problem and improve model performance.

Dataset: diabetes_binary_5050split_health_indicators_BRFSS2015.csv

Target Variable: Diabetes_binary (0 = no diabetes, 1 c= diabetes)

In [2]:
# DX799 Capstone Part 1 - Week 2
# Linear Regression Part 2: Lasso, Ridge, and Elastic Net Regression

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LassoCV, RidgeCV
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

## Load and Prepare Data

Loading the same balanced 50/50 BRFSS dataset from Week 1. 
Splitting into train and test sets before applying regularization. 
Also scaling the features using StandardScaler since Lasso, Ridge, 
and Elastic Net are sensitive to feature scale. Features on 
different scales will get penalized unequally without scaling.

In [3]:
# Load dataset
df = pd.read_csv("diabetes_binary_5050split_health_indicators_BRFSS2015.csv")

# Define features and target
target = 'Diabetes_binary'
feature_cols = [col for col in df.columns if col != target]
X = df[feature_cols]
y = df[target]

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train size:", X_train_scaled.shape)
print("Test size:", X_test_scaled.shape)

Train size: (56553, 21)
Test size: (14139, 21)


## Lasso Regression

Lasso adds an L1 penalty to the regression, which shrinks some 
coefficients to exactly zero, performing automatic feature selection. 
Using LassoCV to find the optimal lambda (alpha) via cross-validation 
across values in logarithmic space. Expecting Lasso to drop features 
that were not statistically significant in Week 1, specifically 
Smoker and PhysActivity.

In [4]:
# Find optimal alpha using cross-validation
alphas = np.logspace(-4, 1, 50)
lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=42)
lasso_cv.fit(X_train_scaled, y_train)

print(f"Best alpha (lambda): {lasso_cv.alpha_:.6f}")

# Train Lasso with best alpha
lasso = Lasso(alpha=lasso_cv.alpha_)
lasso.fit(X_train_scaled, y_train)

# Evaluate
lasso_train_r2 = r2_score(y_train, lasso.predict(X_train_scaled))
lasso_test_r2 = r2_score(y_test, lasso.predict(X_test_scaled))

print(f"Lasso Train R²: {lasso_train_r2:.4f}")
print(f"Lasso Test R²:  {lasso_test_r2:.4f}")

# Show coefficients
lasso_coef = pd.Series(lasso.coef_, index=feature_cols)
print(f"\nFeatures dropped (zero coefficient): {(lasso_coef == 0).sum()}")
print(f"Features kept: {(lasso_coef != 0).sum()}")
print("\nAll coefficients:")
print(lasso_coef.sort_values())

Best alpha (lambda): 0.000518
Lasso Train R²: 0.3100
Lasso Test R²:  0.3072

Features dropped (zero coefficient): 0
Features kept: 21

All coefficients:
HvyAlcoholConsump      -0.024334
Income                 -0.022590
PhysHlth               -0.014789
MentHlth               -0.006099
Education              -0.005331
PhysActivity           -0.003973
Veggies                -0.003381
Fruits                 -0.001052
Smoker                 -0.000545
NoDocbcCost            -0.000144
AnyHealthcare           0.001675
Stroke                  0.006439
DiffWalk                0.010696
HeartDiseaseorAttack    0.017872
Sex                     0.021930
CholCheck               0.024827
HighChol                0.054681
Age                     0.069422
HighBP                  0.076793
BMI                     0.084192
GenHlth                 0.118144
dtype: float64


## Ridge Regression

Ridge adds an L2 penalty which shrinks coefficients toward zero 
but never eliminates them entirely. Useful for handling 
multicollinearity by distributing influence across correlated 
features. Using RidgeCV to find the optimal alpha.

In [5]:
# Find optimal alpha using cross-validation
ridge_alphas = np.logspace(-4, 4, 100)
ridge_cv = RidgeCV(alphas=ridge_alphas, cv=5)
ridge_cv.fit(X_train_scaled, y_train)

print(f"Best alpha (lambda): {ridge_cv.alpha_:.6f}")

# Train Ridge with best alpha
ridge = Ridge(alpha=ridge_cv.alpha_)
ridge.fit(X_train_scaled, y_train)

# Evaluate
ridge_train_r2 = r2_score(y_train, ridge.predict(X_train_scaled))
ridge_test_r2 = r2_score(y_test, ridge.predict(X_test_scaled))

print(f"Ridge Train R²: {ridge_train_r2:.4f}")
print(f"Ridge Test R²:  {ridge_test_r2:.4f}")

# Show coefficients
ridge_coef = pd.Series(ridge.coef_, index=feature_cols)
print("\nAll coefficients:")
print(ridge_coef.sort_values())

Best alpha (lambda): 95.454846
Ridge Train R²: 0.3100
Ridge Test R²:  0.3073

All coefficients:
HvyAlcoholConsump      -0.024664
Income                 -0.023037
PhysHlth               -0.015863
MentHlth               -0.006520
Education              -0.005627
PhysActivity           -0.004342
Veggies                -0.003613
Fruits                 -0.001442
Smoker                 -0.001246
NoDocbcCost            -0.000566
AnyHealthcare           0.002139
Stroke                  0.006829
DiffWalk                0.011296
HeartDiseaseorAttack    0.018144
Sex                     0.022466
CholCheck               0.025176
HighChol                0.054939
Age                     0.069353
HighBP                  0.076755
BMI                     0.084327
GenHlth                 0.118418
dtype: float64


## Elastic Net Regression

Elastic Net blends Lasso and Ridge using two parameters: alpha 
controls overall regularization strength and l1_ratio controls 
the blend between L1 (Lasso) and L2 (Ridge). Testing multiple 
combinations using GridSearchCV to find the optimal settings.

In [6]:
# Elastic Net with GridSearchCV
param_grid = {
    'alpha': np.logspace(-4, 1, 20),
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

enet = ElasticNet(random_state=42, max_iter=10000)
enet_cv = GridSearchCV(enet, param_grid, cv=5, scoring='r2', n_jobs=-1)
enet_cv.fit(X_train_scaled, y_train)

print(f"Best alpha:     {enet_cv.best_params_['alpha']:.6f}")
print(f"Best l1_ratio:  {enet_cv.best_params_['l1_ratio']}")

# Evaluate
enet_train_r2 = r2_score(y_train, enet_cv.predict(X_train_scaled))
enet_test_r2 = r2_score(y_test, enet_cv.predict(X_test_scaled))

print(f"\nElastic Net Train R²: {enet_train_r2:.4f}")
print(f"Elastic Net Test R²:  {enet_test_r2:.4f}")

# Coefficients
enet_coef = pd.Series(enet_cv.best_estimator_.coef_, index=feature_cols)
print(f"\nFeatures dropped: {(enet_coef == 0).sum()}")
print(f"Features kept:    {(enet_coef != 0).sum()}")

Best alpha:     0.000616
Best l1_ratio:  0.7

Elastic Net Train R²: 0.3100
Elastic Net Test R²:  0.3072

Features dropped: 0
Features kept:    21


## Model Comparison Summary

Comparing all three regularization techniques against the Week 1 
baseline. Looking at R-squared, which features each method 
prioritized, and what this tells us about the dataset structure.

In [7]:
# Summary comparison
print("=" * 55)
print("MODEL COMPARISON SUMMARY - WEEKS 1 AND 2")
print("=" * 55)
print(f"{'Model':<25} {'Train R²':<12} {'Test R²':<12}")
print("-" * 55)
print(f"{'Baseline (Week 1)':<25} {'0.309':<12} {'N/A':<12}")
print(f"{'Lasso':<25} {lasso_train_r2:<12.4f} {lasso_test_r2:<12.4f}")
print(f"{'Ridge':<25} {ridge_train_r2:<12.4f} {ridge_test_r2:<12.4f}")
print(f"{'Elastic Net':<25} {enet_train_r2:<12.4f} {enet_test_r2:<12.4f}")
print("=" * 55)
print(f"\nLasso best alpha:        {lasso_cv.alpha_:.6f}")
print(f"Ridge best alpha:        {ridge_cv.alpha_:.6f}")
print(f"Elastic Net best alpha:  {enet_cv.best_params_['alpha']:.6f}")
print(f"Elastic Net l1_ratio:    {enet_cv.best_params_['l1_ratio']}")
print(f"\nFeatures dropped by Lasso:       {(lasso_coef == 0).sum()}")
print(f"Features dropped by Elastic Net: {(enet_coef == 0).sum()}")
print("\nTop 5 features across all models:")
print(lasso_coef.abs().sort_values(ascending=False).head())

MODEL COMPARISON SUMMARY - WEEKS 1 AND 2
Model                     Train R²     Test R²     
-------------------------------------------------------
Baseline (Week 1)         0.309        N/A         
Lasso                     0.3100       0.3072      
Ridge                     0.3100       0.3073      
Elastic Net               0.3100       0.3072      

Lasso best alpha:        0.000518
Ridge best alpha:        95.454846
Elastic Net best alpha:  0.000616
Elastic Net l1_ratio:    0.7

Features dropped by Lasso:       0
Features dropped by Elastic Net: 0

Top 5 features across all models:
GenHlth     0.118144
BMI         0.084192
HighBP      0.076793
Age         0.069422
HighChol    0.054681
dtype: float64


## Key Findings - Week 2

1. All three regularization techniques produced identical R-squared 
   values (~0.307), consistent with the Week 1 baseline of 0.309. 
   The explanatory ceiling for linear models on this dataset is 
   approximately 31%.

2. No features were dropped by Lasso or Elastic Net at their 
   optimal lambda values, suggesting all 21 BRFSS features carry 
   some signal, even if small.

3. GenHlth, BMI, HighBP, Age, and HighChol are consistently the 
   top predictors across all three methods. This cross-method 
   agreement strengthens confidence in these findings.

4. The R-squared ceiling is likely a characteristic of using linear 
   regression on a binary target variable. Logistic regression 
   (Week 4) is expected to be a better fit for this dataset and 
   may produce stronger results.

5. Ridge required a much larger alpha (95.45) compared to Lasso 
   (0.000518) because the squared penalty requires stronger 
   regularization to achieve equivalent shrinkage.